# 07. Resumen de experimento, vairantes y selección de modelo pipeline — `proyecto_integrador_v2`

Este notebook resume la evolución de las versiones del pipeline del proyecto y documenta la selección final del mismo.

Incluye:

1. **V1**: enfoque inicial de clasificación de raza.
2. **V1 ajustada**: mejora de V1 con curaduría, YOLO/crops y mejor preparación visual.
3. **V2**: cambio hacia re-identificación visual con embeddings y similitud coseno.
4. **V2 optimizada**: mejora final con detección/crop optimizados.
5. **Comparación de modelos**: EfficientNetB0 vs EfficientNetV2B0/V2B1.
6. Selección técnica final del pipeline recomendado.

La versión recomendada actual es **V2 optimizada**:

```python
YOLO_MODEL = "yolo26s.pt"
CONF_THRESHOLD = 0.15
CROP_MARGIN = 0.35
EMBEDDING_BACKBONE = "EfficientNetB0"
SIMILARITY = "cosine"
TOP_K = 10
```

# Contexto de versiones y proceso de ajuste


Este documento resume la evolución experimental del proyecto de identificación visual de perros, desde un enfoque inicial de clasificación de razas hasta una versión optimizada orientada a re-identificación visual para casos de perros perdidos/encontrados.

El proyecto inició como un sistema de clasificación de razas, pero evolucionó hacia un sistema de búsqueda visual para perros perdidos/encontrados.

La evolución quedó así:

| Versión | Objetivo | Pregunta que responde | Resultado |
|---|---|---|---|
| **V1** | Clasificar raza | ¿Qué raza parece ser este perro? | Útil, pero insuficiente para identificar perros perdidos |
| **V1 ajustada** | Mejorar la clasificación visual | ¿Podemos mejorar la calidad de entrada y el crop? | Mejor base visual, pero sigue centrada en raza |
| **V2** | Re-identificación visual | ¿Este perro se parece a uno ya reportado? | Mucho más alineada al caso real |
| **V2 optimizada** | Mejorar cobertura y matching | ¿Podemos detectar más perros y mejorar Top-K? | Versión recomendada actual |

Para entender correctamente las comparaciones, es importante distinguir las versiones evaluadas:

## V1 — Clasificación de raza

La primera versión del proyecto se enfocó en resolver un problema de clasificación: identificar la raza de un perro a partir de una imagen.

En esta etapa, el objetivo principal era entrenar y evaluar modelos capaces de responder:

```text
¿Qué raza parece ser este perro?
```

Esta versión permitió construir una primera base técnica usando modelos de visión computacional, métricas de clasificación y preparación de imágenes. Sin embargo, se identificó una limitación importante: reconocer la raza no equivale a identificar al perro. Dos perros de la misma raza pueden ser individuos completamente distintos, y muchos perros perdidos pueden ser mestizos o no corresponder claramente a una raza específica.

Por esa razón, V1 fue útil como punto de partida académico y técnico, pero no era suficiente para resolver el caso real de perros extraviados.

## V1 ajustada — Mejora de la preparación visual

Después de la V1 inicial, se realizaron ajustes para mejorar la calidad visual de las imágenes antes de alimentar los modelos. Esta etapa incluyó procesos de curaduría, revisión de calidad, uso de OpenCV, mejoras de contraste/nitidez y detección/crop del perro mediante YOLO.

El objetivo de esta versión ajustada era responder:

```text
¿Podemos mejorar la entrada visual antes de clasificar o comparar?
```

Esta versión mejoró la limpieza del dataset, la calidad de los crops y la trazabilidad del procesamiento. Sin embargo, aunque la entrada visual mejoró, el enfoque seguía estando principalmente relacionado con clasificación o análisis visual general, no con re-identificación individual.

Por lo tanto, V1 ajustada fue una transición importante entre clasificación de raza y búsqueda visual por identidad.

## V2 — Re-identificación visual

La V2 cambió el enfoque del problema. En lugar de preguntar únicamente por la raza, el sistema comenzó a representar cada perro como un vector visual o embedding.

El objetivo pasó a ser:

```text
¿Este perro se parece visualmente a uno ya reportado?
```

En esta versión, YOLO detecta y recorta al perro, EfficientNetB0 genera embeddings visuales, y la similitud coseno permite buscar los vecinos más parecidos dentro de una base vectorial.

Este cambio fue el más importante a nivel conceptual, porque el problema de perros perdidos/encontrados no se resuelve únicamente con clasificación de raza, sino con recuperación visual de posibles coincidencias.

La V2 permitió evaluar métricas más alineadas con el producto, como:

```text
Top-1 Same Dog Accuracy
Top-5 Same Dog Accuracy
Mean same-dog cosine
Mean different-dog cosine
Separation margin
False positives
False negatives
```

## V2 optimizada — Mejora de detección, crop y búsqueda visual

Después de validar la V2, se identificó que uno de los principales cuellos de botella no estaba en los embeddings, sino en la etapa de detección y crop. Algunas imágenes no llegaban a generar embeddings porque YOLO no detectaba correctamente al perro.

Por eso se probó una configuración optimizada:

```python
CONF_THRESHOLD = 0.15
CROP_MARGIN = 0.35
```

Esta configuración baja el umbral de confianza de YOLO para recuperar más perros y aumenta el margen del crop para conservar más cuerpo y contexto visual.

La V2 optimizada consolidó una nueva línea de notebooks:

```text
02B → 03B → 04B → 05B
```

donde:

```text
02B = detección/crop optimizado
03B = embeddings sobre crops optimizados
04B = búsqueda vectorial optimizada
05B = prueba end-to-end optimizada
```

Esta versión aumentó la cobertura de detección, generó más embeddings útiles y mejoró la métrica principal de re-identificación:

```text
Top-5 Same Dog Accuracy
```

## Comparación de modelos

Además de comparar versiones del pipeline, también se probaron distintos backbones de EfficientNet para evaluar si una arquitectura más nueva mejoraba la re-identificación.

Los modelos comparados fueron:

```text
EfficientNetB0
EfficientNetV2B0
EfficientNetV2B1
```

Aunque EfficientNetV2B1 logró una mejor separación promedio entre perros iguales y perros distintos, EfficientNetB0 obtuvo mejor recuperación Top-5, que es la métrica más relevante para el caso de uso.

Por esa razón, la selección final no se basó únicamente en el modelo más moderno o con mayor separación promedio, sino en el modelo que mejor recupera candidatos útiles para revisión humana.

## Proceso general de ajuste y prueba

El proceso experimental siguió una lógica incremental:

```text
1. Construir una primera versión funcional.
2. Medir sus resultados.
3. Identificar el principal cuello de botella.
4. Ajustar una variable del pipeline.
5. Comparar contra el baseline.
6. Mantener el cambio solo si mejora la métrica relevante.
7. Consolidar la mejor configuración en una nueva versión.
```

En este caso, el cambio más importante fue pasar de clasificación de raza a re-identificación visual. Después, la mejora más efectiva fue optimizar detección y crop, no cambiar a un modelo más grande.

La versión seleccionada actualmente es:

```text
V2 optimizada con YOLO yolo26s.pt, CONF_THRESHOLD 0.15, CROP_MARGIN 0.35 y EfficientNetB0.
```

Esta configuración ofrece el mejor balance entre cobertura, calidad de recuperación visual y utilidad práctica para un sistema de perros perdidos/encontrados.


In [ ]:
# 1. Imports

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)

In [ ]:
# 2. Comparación de versiones: V1, V1 ajustada, V2 y V2 optimizada

version_summary_df = pd.DataFrame([
    {
        "version": "V1",
        "main_goal": "Clasificación de raza",
        "core_question": "¿Qué raza parece ser este perro?",
        "main_method": "EfficientNet como clasificador de razas",
        "main_output": "Raza predicha / Top-5 razas",
        "strength": "Buena base académica para clasificación multiclase",
        "limitation": "La raza no identifica al individuo; dos perros de la misma raza pueden ser distintos"
    },
    {
        "version": "V1 ajustada",
        "main_goal": "Mejorar calidad visual para clasificación",
        "core_question": "¿Podemos mejorar la entrada antes de clasificar?",
        "main_method": "Curaduría visual + OpenCV + YOLO crop + EfficientNet",
        "main_output": "Clasificación con imágenes/crops más limpios",
        "strength": "Mejor control de calidad, mejores crops y dataset más defendible",
        "limitation": "Sigue resolviendo principalmente raza, no identidad"
    },
    {
        "version": "V2",
        "main_goal": "Re-identificación visual",
        "core_question": "¿Este perro se parece visualmente a uno ya reportado?",
        "main_method": "YOLO crop + EfficientNetB0 embeddings + cosine similarity",
        "main_output": "Top-K vecinos visuales",
        "strength": "Alineado al problema real de perros perdidos/encontrados",
        "limitation": "Dependía de la configuración inicial de detección/crop"
    },
    {
        "version": "V2 optimizada",
        "main_goal": "Re-identificación visual con mayor cobertura",
        "core_question": "¿Podemos recuperar más perros sin perder calidad de matching?",
        "main_method": "YOLO conf 0.15 + crop margin 0.35 + EfficientNetB0 embeddings + Top-K",
        "main_output": "Top-K candidatos visuales + decisión visual",
        "strength": "Mejor cobertura, mejor Top-5 Same Dog Accuracy y menos falsos negativos",
        "limitation": "Todavía requiere revisión humana y calibración con más datos reales"
    },
])

display(version_summary_df)

# Último cambio aplicado

El último ajuste importante fue optimizar la detección y el crop.

Configuración base:

```python
CONF_THRESHOLD = 0.25
CROP_MARGIN = 0.15
```

Configuración optimizada:

```python
CONF_THRESHOLD = 0.15
CROP_MARGIN = 0.35
```

Este cambio se consolidó en la línea optimizada:

```text
02B → 03B → 04B → 05B
```

La razón fue simple: bajar el umbral de detección permitió recuperar más perros, y aumentar el margen de crop permitió conservar más información corporal/contextual útil para la re-identificación.

In [ ]:
# 3. Comparación de detección/crop: V2 base vs V2 optimizada

detection_comparison_df = pd.DataFrame([
    {
        "pipeline": "V2 base / Paso 02",
        "yolo_model": "yolo26s.pt",
        "conf_threshold": 0.25,
        "crop_margin": 0.15,
        "images_evaluated": 20580,
        "dogs_detected": 19153,
        "not_detected": 1427,
        "detection_rate_percent": 93.07,
        "crops_ok": 19153,
        "crop_errors": 0,
        "avg_yolo_confidence": 0.8455,
        "multiple_dogs_detected_cases": 2316,
    },
    {
        "pipeline": "V2 optimizada / Paso 02B",
        "yolo_model": "yolo26s.pt",
        "conf_threshold": 0.15,
        "crop_margin": 0.35,
        "images_evaluated": 20580,
        "dogs_detected": 19568,
        "not_detected": 1012,
        "detection_rate_percent": 95.08,
        "crops_ok": 19568,
        "crop_errors": 0,
        "avg_yolo_confidence": 0.8270,
        "multiple_dogs_detected_cases": 2887,
    },
])

detection_comparison_df["extra_crops_vs_base"] = (
    detection_comparison_df["crops_ok"] - detection_comparison_df.loc[0, "crops_ok"]
)

display(detection_comparison_df)

## Lectura

La V2 optimizada recuperó **415 crops adicionales**:

```text
19,153 → 19,568
```

También redujo los casos sin detección:

```text
1,427 → 1,012
```

La confianza promedio de YOLO bajó ligeramente, lo cual es normal al aceptar detecciones con menor umbral. El resultado es positivo porque se recuperaron más imágenes útiles sin errores de crop.

In [ ]:
# 4. Comparación de embeddings: V2 base vs V2 optimizada

embedding_comparison_df = pd.DataFrame([
    {
        "pipeline": "V2 base / Paso 03",
        "input_report": "step02_detection_report.csv",
        "embedding_model": "EfficientNetB0 ImageNet pooling avg",
        "embedding_dim": 1280,
        "embeddings_generated": 19153,
        "embedding_errors": 0,
        "normalization": "L2",
    },
    {
        "pipeline": "V2 optimizada / Paso 03B",
        "input_report": "step02b_detection_report_optimized.csv",
        "embedding_model": "EfficientNetB0 ImageNet pooling avg",
        "embedding_dim": 1280,
        "embeddings_generated": 19568,
        "embedding_errors": 0,
        "normalization": "L2",
    },
])

embedding_comparison_df["extra_embeddings_vs_base"] = (
    embedding_comparison_df["embeddings_generated"] - embedding_comparison_df.loc[0, "embeddings_generated"]
)

display(embedding_comparison_df)

In [ ]:
# 5. Comparación de búsqueda vectorial: Paso 04 vs 04B

search_comparison_df = pd.DataFrame([
    {
        "pipeline": "V2 base / Paso 04",
        "embeddings": 19153,
        "avg_top1_similarity": 0.7589,
        "avg_top5_similarity": 0.7227,
        "avg_top10_similarity": 0.7028,
        "median_top1_similarity": 0.7605,
        "top1_ge_0_90": 854,
        "top1_0_80_0_90": 4754,
        "top1_0_70_0_80": 9267,
        "top1_lt_0_70": 4278,
    },
    {
        "pipeline": "V2 optimizada / Paso 04B",
        "embeddings": 19568,
        "avg_top1_similarity": 0.7489,
        "avg_top5_similarity": 0.7124,
        "avg_top10_similarity": 0.6923,
        "median_top1_similarity": 0.7499,
        "top1_ge_0_90": 819,
        "top1_0_80_0_90": 4165,
        "top1_0_70_0_80": 9374,
        "top1_lt_0_70": 5210,
    },
])

display(search_comparison_df)

## Lectura de búsqueda vectorial

La V2 optimizada tiene más embeddings y cubre más casos difíciles. Por eso baja un poco la similitud promedio.

Esto no significa que el pipeline sea peor: significa que ahora incluye imágenes que antes se quedaban fuera. La métrica más importante para el producto es la de re-identificación, especialmente:

```text
Top-5 Same Dog Accuracy
```

In [ ]:
# 6. Comparación de re-identificación: V2 base vs V2 optimizada

reid_comparison_df = pd.DataFrame([
    {
        "pipeline": "V2 base",
        "conf_threshold": 0.25,
        "crop_margin": 0.15,
        "valid_crops": 371,
        "no_dog_detected": 157,
        "num_dogs": 97,
        "top1_same_dog_accuracy": 0.9218,
        "top5_same_dog_accuracy": 0.9569,
        "top10_same_dog_accuracy": 0.9596,
        "mean_same_dog_cosine": 0.6835,
        "mean_different_dog_cosine": 0.2261,
        "separation_margin": 0.4574,
        "false_negatives_top5": 16,
        "false_positives_top1": 29,
    },
    {
        "pipeline": "V2 optimizada",
        "conf_threshold": 0.15,
        "crop_margin": 0.35,
        "valid_crops": 412,
        "no_dog_detected": 116,
        "num_dogs": 97,
        "top1_same_dog_accuracy": 0.9320,
        "top5_same_dog_accuracy": 0.9709,
        "top10_same_dog_accuracy": 0.9806,
        "mean_same_dog_cosine": 0.6805,
        "mean_different_dog_cosine": 0.2253,
        "separation_margin": 0.4552,
        "false_negatives_top5": 12,
        "false_positives_top1": 28,
    },
    {
        "pipeline": "V2 optimizada alternativa",
        "conf_threshold": 0.15,
        "crop_margin": 0.25,
        "valid_crops": 412,
        "no_dog_detected": 116,
        "num_dogs": 97,
        "top1_same_dog_accuracy": 0.9296,
        "top5_same_dog_accuracy": 0.9709,
        "top10_same_dog_accuracy": 0.9806,
        "mean_same_dog_cosine": 0.6802,
        "mean_different_dog_cosine": 0.2251,
        "separation_margin": 0.4551,
        "false_negatives_top5": 12,
        "false_positives_top1": 29,
    },
])

for col in ["top1_same_dog_accuracy", "top5_same_dog_accuracy", "top10_same_dog_accuracy"]:
    reid_comparison_df[col + "_percent"] = reid_comparison_df[col] * 100

display(reid_comparison_df)

## Lectura de re-identificación

La V2 optimizada mejoró la métrica principal:

```text
Top-5 Same Dog Accuracy: 95.69% → 97.09%
```

También mejoró:

```text
Top-1 Same Dog Accuracy: 92.18% → 93.20%
Top-10 Same Dog Accuracy: 95.96% → 98.06%
False negatives Top-5: 16 → 12
False positives Top-1: 29 → 28
```

La variante `0.15 / 0.35` fue ligeramente mejor que `0.15 / 0.25`, por lo que se selecciona como configuración recomendada.

# Comparación de modelos EfficientNet

Se probaron variantes de EfficientNet para ver si una arquitectura más nueva mejoraba la re-identificación.

La comparación mostró que EfficientNetV2B1 mejora la separación promedio, pero no mejora la métrica principal de producto: **Top-5 Same Dog Accuracy**.

Por eso, el modelo recomendado sigue siendo **EfficientNetB0**.

In [ ]:
# 7. Comparación de backbones EfficientNet

model_comparison_df = pd.DataFrame([
    {
        "backbone": "EfficientNetB0",
        "top1_same_dog_accuracy": 0.9218,
        "top5_same_dog_accuracy": 0.9569,
        "top10_same_dog_accuracy": 0.9596,
        "mean_same_dog_cosine": 0.6835,
        "mean_different_dog_cosine": 0.2261,
        "separation_margin": 0.4574,
        "false_negatives_top5": 16,
        "false_positives_top1": 29,
        "decision": "Modelo principal recomendado"
    },
    {
        "backbone": "EfficientNetV2B0",
        "top1_same_dog_accuracy": 0.9084,
        "top5_same_dog_accuracy": 0.9542,
        "top10_same_dog_accuracy": 0.9569,
        "mean_same_dog_cosine": 0.7005,
        "mean_different_dog_cosine": 0.2193,
        "separation_margin": 0.4811,
        "false_negatives_top5": 17,
        "false_positives_top1": 34,
        "decision": "No mejora Top-5; no recomendado como principal"
    },
    {
        "backbone": "EfficientNetV2B1",
        "top1_same_dog_accuracy": 0.9218,
        "top5_same_dog_accuracy": 0.9515,
        "top10_same_dog_accuracy": 0.9569,
        "mean_same_dog_cosine": 0.7103,
        "mean_different_dog_cosine": 0.2152,
        "separation_margin": 0.4951,
        "false_negatives_top5": 18,
        "false_positives_top1": 29,
        "decision": "Mejor separación, pero peor Top-5 que B0"
    },
])

for col in ["top1_same_dog_accuracy", "top5_same_dog_accuracy", "top10_same_dog_accuracy"]:
    model_comparison_df[col + "_percent"] = model_comparison_df[col] * 100

display(model_comparison_df)

## Lectura de modelos

EfficientNetV2B1 logró mejor separación promedio:

```text
Separation margin: 0.4951
```

pero EfficientNetB0 tuvo mejor recuperación Top-5:

```text
EfficientNetB0 Top-5: 95.69%
EfficientNetV2B1 Top-5: 95.15%
```

Como el objetivo del producto es mostrar candidatos correctos para revisión humana, la métrica más importante es **Top-5 Same Dog Accuracy**.  
Por eso se selecciona **EfficientNetB0** como backbone principal.

# Selección final del pipeline

La versión recomendada es:

```text
V2 optimizada
```

Configuración final:

```python
YOLO_MODEL = "yolo26s.pt"
CONF_THRESHOLD = 0.15
CROP_MARGIN = 0.35
EMBEDDING_BACKBONE = "EfficientNetB0"
EMBEDDING_DIM = 1280
NORMALIZATION = "L2"
SIMILARITY = "cosine"
TOP_K = 10
```

Pipeline final:

```text
01_Image_Quality_Curation
↓
02B_Dog_Detection_Cropping_YOLO_Optimized
↓
03B_Dog_Visual_Embeddings_Optimized
↓
04B_Vector_Search_Cosine_Similarity_Optimized
↓
05B_Lost_Found_End_to_End_Search_Optimized
↓
06_Dog_ReIdentification_Evaluation
↓
07_Experiment_Summary_and_Model_Selection
```

In [ ]:
# 8. Configuración final seleccionada

final_selection_df = pd.DataFrame([
    {"component": "Pipeline version", "selected_value": "V2 optimizada"},
    {"component": "YOLO model", "selected_value": "yolo26s.pt"},
    {"component": "Confidence threshold", "selected_value": 0.15},
    {"component": "Crop margin", "selected_value": 0.35},
    {"component": "Embedding backbone", "selected_value": "EfficientNetB0"},
    {"component": "Embedding dimension", "selected_value": 1280},
    {"component": "Normalization", "selected_value": "L2"},
    {"component": "Similarity metric", "selected_value": "cosine similarity"},
    {"component": "Retrieval", "selected_value": "Top-K neighbors"},
    {"component": "Recommended Top-K", "selected_value": 10},
    {"component": "Main product metric", "selected_value": "Top-5 Same Dog Accuracy"},
])

display(final_selection_df)

# Limitaciones

Aunque la V2 optimizada es la versión recomendada, todavía existen limitaciones:

1. Los matches visuales no deben interpretarse como certeza absoluta.
2. El sistema debe mostrar candidatos para revisión humana.
3. Las imágenes con múltiples perros requieren manejo especial.
4. La detección puede fallar en perros pequeños, parciales o con mala iluminación.
5. El dataset de identidad todavía debe crecer para validar generalización.
6. La similitud visual debe combinarse con metadata real: ubicación, fecha, tamaño, color, descripción y contexto del reporte.

# Próximos pasos recomendados

Los siguientes pasos serían:

1. **Agrupar resultados por `dog_id`** en lugar de mostrar solo vecinos individuales.
2. Implementar **multi-embedding por perro**: varias fotos por identidad.
3. Calibrar umbrales reales de match fuerte / posible / débil.
4. Revisar manualmente falsos positivos y falsos negativos.
5. Agregar metadata contextual: ubicación, fecha, color, collar, tamaño, descripción.
6. Probar modelos especializados para embeddings: DINOv2, CLIP o Siamese/Triplet.
7. Entrenar una red Siamese/Triplet cuando haya suficientes identidades reales.
8. Construir demo web/app para flujo lost/found.

# Conclusión final

La evolución del proyecto muestra que la solución más adecuada no es clasificar razas, sino hacer re-identificación visual.

La **V1** fue útil como base de clasificación.  
La **V1 ajustada** mejoró la calidad visual y el procesamiento de imágenes.  
La **V2** replanteó correctamente el problema hacia embeddings y similitud visual.  
La **V2 optimizada** mejoró la cobertura y el desempeño de re-identificación.

Por lo tanto, la versión recomendada actual es:

```text
V2 optimizada con YOLO yolo26s.pt, CONF_THRESHOLD 0.15, CROP_MARGIN 0.35 y EfficientNetB0.
```

Esta versión ofrece el mejor balance entre cobertura, recuperación visual y utilidad práctica para un sistema de perros perdidos/encontrados.